# Ancient Tamil Palm-Leaf Character Recognition (Poornima Devi & Sornam, 2020)
## Modified Adaptive Backpropagation Neural Network (MABPN) with GLCM Features
### ACM Transactions on Asian and Low-Resource Language Information Processing (ACM TALLIP)

---

This notebook provides an interactive walkthrough of the exact research paper implementation:
1. **Preprocessing:** Grayscale -> Median Filter -> Adaptive Mean Thresholding -> Convex Hull Segmentation -> Uniform 8x8 Resizing.
2. **Model 1: MABPN-BCV (65-55-5):** 64-D binary coded value input, 55 hidden units, 5 output units.
3. **Model 2: MABPN-GLCM (6-20-5):** Exactly 6 GLCM features (Contrast, Energy, Homogeneity, Correlation, Dissimilarity, ASM), 20 hidden units, 5 output units.
4. **Mathematics:** Nguyen-Widrow weight initialization (Eq. 1-6), Sigmoid (Eq. 7), Shannon Wavelet activation (Eq. 8), and MABPN linear/non-linear error updates (Eq. 9-20).
5. **Figure 12 Mapping:** 5-bit codeword encoding and decoding for the 18 Tamil consonants (*Meyyezhuthu*).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Ensure UTF-8 output encoding on Windows
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = ['Nirmala UI', 'Latha', 'Vijaya', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

import config
import dataset
import preprocess
import features
import neural_networks
import models
import evaluate
import predict
import process_manuscript

print('All paper modules successfully imported!')

## 1. Dataset Verification & Loading (70:30 Stratified Split)
Loads the 18 Tamil consonant classes (*Meyyezhuthu*) from Figure 12.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test, class_names, status = dataset.load_character_dataset(
    data_dir=config.DATA_DIR,
    test_size=config.TRAIN_TEST_SPLIT_RATIO,
    random_state=config.RANDOM_SEED,
    allow_test_fixture=True
)
print(f'Loaded {len(class_names)} canonical classes: {class_names}')
print(f'Training set (70%): {len(X_train_raw)} samples')
print(f'Testing set (30%):  {len(X_test_raw)} samples')

## 2. Preprocessing & Feature Extraction
Extracts 64-D Binary Coded Values for Model 1 and 6-D GLCM features for Model 2.

In [ ]:
# Model 1: 64-D Binary Coded Values
X_tr_bcv = features.extract_features_batch(X_train_raw, model_type='bcv')
X_te_bcv = features.extract_features_batch(X_test_raw, model_type='bcv')
print('MABPN-BCV Feature Shape:', X_tr_bcv.shape)

# Model 2: 6-D GLCM Features
X_tr_glcm = features.extract_features_batch(X_train_raw, model_type='glcm')
X_te_glcm = features.extract_features_batch(X_test_raw, model_type='glcm')
print('MABPN-GLCM Feature Shape:', X_tr_glcm.shape)

## 3. Train & Evaluate Model 1: MABPN-BCV (65-55-5)

In [ ]:
clf_bcv = models.MABPN_BCV(learning_rate=0.05, max_iter=130, random_state=config.RANDOM_SEED)
clf_bcv.fit(X_tr_bcv, y_train)
y_pred_bcv = clf_bcv.predict(X_te_bcv)
raw_bcv = clf_bcv.predict_bits(X_te_bcv)
metrics_bcv = evaluate.compute_paper_metrics(y_test, y_pred_bcv, class_names, model=clf_bcv, raw_outputs=raw_bcv)
print(evaluate.format_evaluation_summary('MABPN-BCV', metrics_bcv, len(clf_bcv.loss_history_), clf_bcv.loss_history_[-1]))
evaluate.plot_paper_confusion_matrix(metrics_bcv['confusion_matrix'], class_names, model_name='MABPN-BCV')

## 4. Train & Evaluate Model 2: MABPN-GLCM (6-20-5)

In [ ]:
clf_glcm = models.MABPN_GLCM(learning_rate=0.05, max_iter=104, random_state=config.RANDOM_SEED)
clf_glcm.fit(X_tr_glcm, y_train)
y_pred_glcm = clf_glcm.predict(X_te_glcm)
raw_glcm = clf_glcm.predict_bits(X_te_glcm)
metrics_glcm = evaluate.compute_paper_metrics(y_test, y_pred_glcm, class_names, model=clf_glcm, raw_outputs=raw_glcm)
print(evaluate.format_evaluation_summary('MABPN-GLCM', metrics_glcm, len(clf_glcm.loss_history_), clf_glcm.loss_history_[-1]))
evaluate.plot_paper_confusion_matrix(metrics_glcm['confusion_matrix'], class_names, model_name='MABPN-GLCM')

## 5. End-to-End Manuscript Strip Processing
Restores, segments, and classifies characters from an authentic palm-leaf manuscript image.

In [ ]:
process_manuscript.process_manuscript_strip(
    manuscript_path=str(config.MANUSCRIPTS_DIR / 'tolkappiyam_palm_leaf.jpg'),
    output_dir=str(config.RESULTS_DIR / 'manuscript_crops'),
    max_crops=10
)